## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../../src").resolve()))
from condpred import report
from condpred.experiments import e7

SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
OUT = Path("../../notebooks/tables/condpred/E7" + ("/smoke" if SMOKE else ""))
FIG = OUT / "figures"
CACHE = "E7/smoke" if SMOKE else "E7"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

SPEC = e7.build_initial_spec(SMOKE)

## 2. Write spec

In [ ]:
report.write_spec(SPEC, OUT, modules=e7.E7_MODULES, completion_keys=e7.COMPLETION_KEYS)
e7.verify_spec_immutable(OUT, SPEC, e7.E7_MODULES)

## 3. Leakage test

In [ ]:
leaks = e7.run_leakage_e7()
ldf = pd.DataFrame(leaks, columns=["pipeline", "pass"])
report.save_table(ldf, OUT, "leakage")
print(ldf.to_string(index=False))
assert ldf["pass"].all(), ldf[~ldf["pass"]]

## 4. Load E4 snapshot, month-end grid

In [ ]:
G2, G2_NOTE = e7.g2_status()
GATE_SKIP = (not SMOKE) and G2 != "pass"
if GATE_SKIP:
    DATA = None
    print("G2 gate:", G2, "-", G2_NOTE, "; skipping load/panel (heavy cells)")
else:
    DATA = e7.load_prices(SPEC)
    SPEC["snapshot"] = DATA["snapshot"]
    SPEC["universes"] = {"assets": sorted(DATA["prices"]), "kept_tier_a": DATA["kept"]}
    report.write_spec(SPEC, OUT, modules=e7.E7_MODULES, completion_keys=e7.COMPLETION_KEYS)
    print("snapshot:", DATA["snapshot"])
    print("assets:", sorted(DATA["prices"]))

## 5. TSMOM positions and targets

In [ ]:
if GATE_SKIP:
    PANEL_RAW = None
    print("skipped (G2 gate)")
else:
    PANEL_RAW = e7.build_panel(SPEC, DATA, CACHE)
    print(PANEL_RAW.groupby("asset").size().to_string())
    print(PANEL_RAW[["hit", "strat_ret"]].describe().to_string())

## 6. Features

In [ ]:
if GATE_SKIP:
    print("skipped (G2 gate)")
else:
    GROUPS = e7.feature_groups(PANEL_RAW.columns)
    for k, v in GROUPS.items():
        print(k, len(v), v)

## 7. Walk-forward

In [ ]:
if GATE_SKIP:
    print("skipped (G2 gate)")
else:
    print("first_test_year:", SPEC["first_test_year"], "panel:", PANEL_RAW["date"].min(), PANEL_RAW["date"].max())

## 8. Decision

In [ ]:
RESULT = e7.run_pipeline(SPEC, OUT, FIG, CACHE, data=DATA)
print(pd.read_csv(OUT / "decision.csv").to_string(index=False))
if RESULT.get("skipped"):
    print("pipeline skipped:", RESULT.get("g2"))
else:
    print("primary:", RESULT["primary"])

## 9. Secondary

In [ ]:
for name in ("secondary_results", "secondary_off_switch", "per_asset"):
    p = OUT / f"{name}.csv"
    if p.exists():
        print(name)
        print(pd.read_csv(p).to_string(index=False))

## 10. Figures/tables

In [ ]:
print(sorted(p.name for p in OUT.glob("*.csv")))
print(sorted(p.name for p in FIG.glob("*.png")))